# RunPod ComfyUI end-to-end workflow

This notebook loads an API-format ComfyUI workflow and submits one job per CSV row to a RunPod Serverless endpoint. The CSV must contain `Image_URL` and `Video_URL` columns. Results are saved beside the input as `output_for_<input filename>`.

In [ ]:
import copy
import json
import os
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / 'tests' / 'e2e').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the project directory or one of its subdirectories.')
WORKFLOW_PATH = PROJECT_ROOT / 'tests' / 'workflow' / 'video_wan2_2_14B_animate .json'

RUNPOD_API_KEY = os.environ['RUNPOD_API_KEY']
RUNPOD_ENDPOINT_ID = os.environ['RUNPOD_ENDPOINT_ID']
RUNPOD_API_BASE = os.getenv('RUNPOD_API_BASE', 'https://api.runpod.ai/v2')
REQUEST_TIMEOUT_SECONDS = int(os.getenv('E2E_TIMEOUT_SECONDS', '1800'))
BULK_CSV_PATH = Path(os.getenv('BULK_CSV_PATH', str(PROJECT_ROOT / 'tests' / 'e2e' / 'bulk_inputs.csv')))

In [ ]:
with WORKFLOW_PATH.open(encoding='utf-8') as workflow_file:
    base_workflow = json.load(workflow_file)

if not isinstance(base_workflow, dict) or not base_workflow:
    raise ValueError(f'Expected a non-empty API workflow: {WORKFLOW_PATH}')

print(f'Loaded {len(base_workflow)} workflow nodes from {WORKFLOW_PATH.name}')

In [ ]:
def build_workflow(workflow: dict, payload: dict) -> dict:
    """Apply request parameters without modifying the loaded workflow."""
    updated_workflow = copy.deepcopy(workflow)
    image_nodes = 0
    video_nodes = 0

    for node in updated_workflow.values():
        if not isinstance(node, dict):
            continue
        inputs = node.get('inputs')
        if not isinstance(inputs, dict):
            continue

        if node.get('class_type') == 'LoadImage':
            inputs['image'] = payload['image_url']
            image_nodes += 1
        elif node.get('class_type') == 'LoadVideo':
            inputs['file'] = payload['video_url']
            video_nodes += 1

    if image_nodes == 0 or video_nodes == 0:
        raise ValueError('This notebook requires a workflow with LoadImage and LoadVideo nodes.')

    for node_id, input_updates in payload.get('node_inputs', {}).items():
        node = updated_workflow.get(str(node_id))
        if not isinstance(node, dict) or not isinstance(node.get('inputs'), dict):
            raise KeyError(f'Workflow node {node_id} does not have an inputs object.')
        node['inputs'].update(input_updates)

    return updated_workflow

In [ ]:
import csv
from pathlib import Path


def read_csv(csv_path):
    with Path(csv_path).open(newline='', encoding='utf-8-sig') as csv_file:
        reader = csv.DictReader(csv_file)
        required_columns = {'Image_URL', 'Video_URL'}
        if not reader.fieldnames or not required_columns.issubset(reader.fieldnames):
            raise ValueError('CSV must contain Image_URL and Video_URL columns.')

        rows = []
        for row_number, row in enumerate(reader, start=2):
            image_url = (row.get('Image_URL') or '').strip()
            video_url = (row.get('Video_URL') or '').strip()
            if not image_url or not video_url:
                raise ValueError(f'CSV row {row_number} is missing an image or video URL.')
            rows.append((row_number, image_url, video_url))
    return rows

In [ ]:
import csv


def write_to_csv(output_path, rows):
    fieldnames = ['Image_URL', 'Video_URL', 'status', 'error', 'node_id', 'output_key', 'filename', 'url']
    with open(output_path, 'w', newline='', encoding='utf-8') as output_file:
        writer = csv.DictWriter(output_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

In [ ]:
import asyncio
import csv

import aiohttp


class RunpodServerlessClient:
    def __init__(self, api_key, endpoint_id, api_base=RUNPOD_API_BASE, timeout=REQUEST_TIMEOUT_SECONDS, backoff_factor=1, max_retries=3):
        self.endpoint_url = f'{api_base.rstrip("/")}/{endpoint_id}/runsync'
        self.headers = {'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'}
        self.timeout = aiohttp.ClientTimeout(total=timeout)
        self.backoff_factor = backoff_factor
        self.max_retries = max_retries
        self.session = None

    async def __aenter__(self):
        self.session = aiohttp.ClientSession(headers=self.headers, timeout=self.timeout)
        return self

    async def __aexit__(self, exc_type, exc, traceback):
        await self.session.close()

    async def create_one_task(self, payload):
        for attempt in range(self.max_retries + 1):
            try:
                async with self.session.post(self.endpoint_url, json=payload) as response:
                    if response.status == 429 or response.status >= 500:
                        if attempt == self.max_retries:
                            response.raise_for_status()
                        delay = self.backoff_factor * (2 ** attempt)
                        print(f'RunPod returned {response.status}; retrying in {delay}s')
                    else:
                        response.raise_for_status()
                        return await response.json()
            except (aiohttp.ClientError, asyncio.TimeoutError):
                if attempt == self.max_retries:
                    raise
                delay = self.backoff_factor * (2 ** attempt)
            await asyncio.sleep(delay)

    async def create_bulk_tasks(self, csv_path, base_workflow):
        input_path = Path(csv_path)
        output_path = input_path.with_name(f'output_for_{input_path.name}')
        if output_path.resolve() == input_path.resolve():
            raise ValueError('Output CSV path must not overwrite the input CSV.')
        output_rows = []
        tasks = []
        task_inputs = []

        async def iter_csv_rows(rows):
            for row in rows:
                yield row
                await asyncio.sleep(0)

        for row_number, image_url, video_url in iter_csv_rows(read_csv(csv_path)):
            workflow = build_workflow(base_workflow, {'image_url': image_url, 'video_url': video_url})
            task_payload = {'input': {'workflow': workflow}}
            tasks.append(asyncio.create_task(self.create_one_task(task_payload)))
            task_inputs.append((row_number, image_url, video_url))

        task_results = await asyncio.gather(*tasks, return_exceptions=True)
        for (row_number, image_url, video_url), task_result in zip(task_inputs, task_results):
            if isinstance(task_result, BaseException):
                output_rows.append({'Image_URL': image_url, 'Video_URL': video_url, 'status': 'failed', 'error': str(task_result)})
                continue
            if not isinstance(task_result, dict):
                output_rows.append({'Image_URL': image_url, 'Video_URL': video_url, 'status': 'failed', 'error': f'Unexpected response for CSV row {row_number}: {task_result}'})
                continue
            handler_result = task_result.get('output', {})
            if not isinstance(handler_result, dict) or task_result.get('status') != 'COMPLETED' or handler_result.get('status') != 'COMPLETED':
                output_rows.append({'Image_URL': image_url, 'Video_URL': video_url, 'status': 'failed', 'error': f'CSV row {row_number}: {task_result}'})
                continue
            generated_outputs = handler_result.get('outputs', [])
            if not generated_outputs:
                output_rows.append({'Image_URL': image_url, 'Video_URL': video_url, 'status': 'successful', 'error': ''})
            for generated_output in generated_outputs:
                output_rows.append({
                    'Image_URL': image_url,
                    'Video_URL': video_url,
                    'status': 'successful',
                    'error': '',
                    'node_id': generated_output.get('node_id', ''),
                    'output_key': generated_output.get('output_key', ''),
                    'filename': generated_output.get('filename', ''),
                    'url': generated_output.get('url', ''),
                })

        write_to_csv(output_path, output_rows)
        print(f'Saved {len(output_rows)} job result row(s) to {output_path}')
        return output_path


async with RunpodServerlessClient(RUNPOD_API_KEY, RUNPOD_ENDPOINT_ID) as runpod_client:
    output_csv_path = await runpod_client.create_bulk_tasks(BULK_CSV_PATH, base_workflow)

print(f'Bulk job results saved to: {output_csv_path}')